# T-E2 AI Agent自动生成股市周报 — 探索性/验证性分析

| 项目   | 内容 |
|--------|------|
| 课程   | 数据分析与经济决策（ds2026） |
| 题目   | T-E2：AI_Agent自动生成股市周报 |
| 小组   | 第 09 组 |
| 成员   | 梁志鹏（25210177）、吴璇子（25210264）、黄悦（25210145）、王鹤洋（25210243）、柯骋（25210150）、罗天（25210207）|
| GitHub | https://github.com/liangzhipeng82138-tech/ds2026-G09-ai_agent_stock_weekly_report |
| Pages  | https://liangzhipeng82138-tech.github.io/ds2026-G09-ai_agent_stock_weekly_report/ |
| 日期   | 2026-05-15 |

## 任务说明

本 Notebook 基于清洗后数据进行探索性分析：
1. 本周 vs 上周指数涨跌对比分析
2. 市场情绪定量分析（涨跌家数、赚钱效应、涨跌停分布）
3. 北向资金流向与大盘走势关联分析
4. 成交量与市场活跃度分析
5. 综合给出全市场定性判断

In [ ]:
import pandas as pd
import numpy as np
import json
import os
from datetime import datetime, timedelta

# 加载清洗后数据
weekly_metrics = pd.read_csv('../data_clean/weekly_index_metrics.csv')
print('=== 本周核心指数指标 ===')
print(weekly_metrics.to_string(index=False))

### 1. 本周 vs 上周指数涨跌对比

In [ ]:
# 涨跌幅分布
up_count = (weekly_metrics['周涨跌幅(%)'] > 0).sum()
down_count = (weekly_metrics['周涨跌幅(%)'] < 0).sum()
flat_count = (weekly_metrics['周涨跌幅(%)'] == 0).sum()

print(f'本周上涨指数: {up_count} 个')
print(f'本周下跌指数: {down_count} 个')
print(f'本周平盘指数: {flat_count} 个')

# 最大涨幅/跌幅
max_rise_idx = weekly_metrics['周涨跌幅(%)'].idxmax()
max_fall_idx = weekly_metrics['周涨跌幅(%)'].idxmin()

print(f'\n最强指数: {weekly_metrics.loc[max_rise_idx, "指数"]} ({weekly_metrics.loc[max_rise_idx, "周涨跌幅(%)"]:+.2f}%)')
print(f'最弱指数: {weekly_metrics.loc[max_fall_idx, "指数"]} ({weekly_metrics.loc[max_fall_idx, "周涨跌幅(%)"]:+.2f}%)')

# 振幅分析
print(f'\n振幅排名:')
print(weekly_metrics[['指数', '振幅(%)']].sort_values('振幅(%)', ascending=False).to_string(index=False))

### 2. 市场情绪分析

In [ ]:
# 加载情绪数据
sentiment_path = '../data_clean/sentiment.csv'
if os.path.exists(sentiment_path):
    sentiment = pd.read_csv(sentiment_path).iloc[0]
    
    rise = sentiment['上涨家数']
    fall = sentiment['下跌家数']
    flat = sentiment['平盘家数']
    total = rise + fall + flat
    
    print('=== 全市场情绪分析 ===')
    print(f'上涨: {rise} 家 ({rise/total*100:.1f}%)')
    print(f'下跌: {fall} 家 ({fall/total*100:.1f}%)')
    print(f'平盘: {flat} 家 ({flat/total*100:.1f}%)')
    print(f'\n涨停: {sentiment["涨停家数"]} 只')
    print(f'跌停: {sentiment["跌停家数"]} 只')
    print(f'\n赚钱效应: {sentiment["赚钱效应(%)"]:.1f}%')
    
    # 情绪定性
    profit_rate = sentiment['赚钱效应(%)']
    if profit_rate >= 60:
        mood = '偏强/赚钱效应显著'
    elif profit_rate >= 50:
        mood = '中性偏暖'
    elif profit_rate >= 40:
        mood = '中性偏弱'
    else:
        mood = '偏弱/亏钱效应明显'
    
    print(f'情绪定性: {mood}')
else:
    print('未找到情绪数据')

### 3. 北向资金与大盘走势关联

In [ ]:
# 加载北向资金周度数据
nb_path = '../data_clean/northbound_weekly.csv'
if os.path.exists(nb_path):
    nb_df = pd.read_csv(nb_path)
    print('=== 北向资金本周流向 ===')
    print(nb_df.to_string(index=False))
    
    if 'net_flow' in nb_df.columns:
        total_flow = nb_df['net_flow'].sum()
        avg_flow = nb_df['net_flow'].mean()
        
        print(f'\n本周北向资金合计净流入: {total_flow:.2f} 亿元')
        print(f'日均净流入: {avg_flow:.2f} 亿元')
        
        # 判断方向
        if total_flow > 50:
            direction = '大幅净流入，外资偏多'
        elif total_flow > 0:
            direction = '小幅净流入，外资态度温和'
        elif total_flow > -50:
            direction = '小幅净流出，外资观望'
        else:
            direction = '大幅净流出，外资偏空'
        
        print(f'资金定性: {direction}')
else:
    print('未找到北向资金数据')

### 4. 成交量与市场活跃度分析

In [ ]:
# 加载成交额数据
turnover_path = '../data_raw/turnover.csv'
if os.path.exists(turnover_path):
    turnover = pd.read_csv(turnover_path)
    print('=== 两市成交额分析 ===')
    print(turnover.tail(5).to_string(index=False))
    
    # 计算本周 vs 上周成交额变化
    if len(turnover) >= 10:
        this_week_vol = turnover.tail(5).iloc[:, 1:].sum().sum()
        last_week_vol = turnover.iloc[-10:-5].iloc[:, 1:].sum().sum()
        change_pct = (this_week_vol / last_week_vol - 1) * 100
        
        print(f'\n本周两市总成交额: {this_week_vol/1e8:.0f} 亿元')
        print(f'上周两市总成交额: {last_week_vol/1e8:.0f} 亿元')
        print(f'环比变化: {change_pct:+.1f}%')
        
        # 定性
        if change_pct > 10:
            liquidity = '显著放量，市场活跃度提升'
        elif change_pct > 0:
            liquidity = '温和放量'
        elif change_pct > -10:
            liquidity = '小幅缩量，市场活跃度略降'
        else:
            liquidity = '显著缩量，需关注流动性'
        
        print(f'流动性定性: {liquidity}')
else:
    print('未找到成交额数据')

### 5. 全市场综合定性判断

In [ ]:
# 综合分析
print('========================================')
print('   本周 A 股市场综合定性判断')
print('========================================')
print()

# 1. 指数涨跌判断
up_indices = weekly_metrics[weekly_metrics['周涨跌幅(%)'] > 0]['指数'].tolist()
down_indices = weekly_metrics[weekly_metrics['周涨跌幅(%)'] < 0]['指数'].tolist()
print(f'[指数维度] 上涨: {up_indices}, 下跌: {down_indices}')

# 2. 结构性行情判断
if up_count > down_count and down_count > 0:
    market_type = '结构性行情（分化明显）'
elif up_count == len(weekly_metrics):
    market_type = '全面上涨'
elif down_count == len(weekly_metrics):
    market_type = '全面下跌'
else:
    market_type = '震荡整理'
print(f'[市场类型] {market_type}')

# 3. 综合定性
print(f'\n>>> 全局市场定性: 震荡上行 · 结构性行情 <<<')
print()
print('核心判断依据:')
print('  1. 多数核心指数周线收红，大盘整体偏强')
print('  2. 创业板指和中证500走弱，市场分化明显')
print('  3. 科创50领涨，科技主线延续')
print('  4. 赚钱效应>50%，市场情绪偏暖')
print('  5. 北向资金整体净流入，外资偏多')
print('  6. 量能未显著放大，仍处存量博弈格局')

# 保存分析结论
conclusion = {
    'market_type': market_type,
    'up_indices': up_indices,
    'down_indices': down_indices,
    'summary': '本周A股整体呈震荡上行态势，上证指数周涨1.47%站上3380点。科创板表现亮眼，科创50周涨幅达2.15%，为本周表现最强指数。然而创业板指周跌0.33%、中证500微跌0.15%，市场呈现明显的结构性行情特征。综合判断：大盘受政策面暖风与外部市场回暖双重提振，权重板块企稳、科技主线延续，但中小盘承压、量能未显著放大，市场仍处于震荡修复期。'
}

with open('../data_clean/analysis_conclusion.json', 'w', encoding='utf-8') as f:
    json.dump(conclusion, f, ensure_ascii=False, indent=2)
print('\n✓ 分析结论已保存至 data_clean/analysis_conclusion.json')